# Error Analysis for Robinson Crusoe Adaptation Detection

This notebook performs a deep analysis of misclassified examples to understand:
- Why certain texts were misclassified
- Common patterns in errors
- Text characteristics of false positives/negatives
- Embedding similarity of error cases

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import tensorflow_hub as hub
from sklearn.metrics.pairwise import cosine_similarity
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

print("Environment ready")

## 1. Load Misclassified Examples

In [ ]:
# Load misclassified examples from training notebook
misclass_df = pd.read_csv('misclassified_examples.csv')

print(f"Total misclassified examples: {len(misclass_df)}")
print(f"\nBreakdown:")
print(f"  False Positives (Random → RC): {len(misclass_df[misclass_df['actual_class'] == 0])}")
print(f"  False Negatives (RC → Random): {len(misclass_df[misclass_df['actual_class'] == 1])}")

print(f"\nDataset info:")
print(misclass_df.info())

print(f"\nFirst few examples:")
print(misclass_df[['actual_class', 'predicted_class', 'confidence', 'prob_random', 'prob_rc']].head())

## 2. Text Characteristics Analysis

In [ ]:
# Analyze text characteristics
misclass_df['text_length'] = misclass_df['text'].apply(len)
misclass_df['word_count'] = misclass_df['text'].apply(lambda x: len(str(x).split()))
misclass_df['avg_word_length'] = misclass_df['text'].apply(
    lambda x: np.mean([len(word) for word in str(x).split()]) if len(str(x).split()) > 0 else 0
)

print("\nText Characteristics of Misclassified Examples:")
print("=" * 70)
print(f"\nText Length Statistics:")
print(misclass_df.groupby('actual_class')['text_length'].describe())

print(f"\nWord Count Statistics:")
print(misclass_df.groupby('actual_class')['word_count'].describe())

print(f"\nAverage Word Length:")
print(misclass_df.groupby('actual_class')['avg_word_length'].describe())

In [ ]:
# Visualize characteristics
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Text length by error type
for class_idx in misclass_df['actual_class'].unique():
    data = misclass_df[misclass_df['actual_class'] == class_idx]['text_length']
    label = 'False Positive' if class_idx == 0 else 'False Negative'
    axes[0, 0].hist(data, bins=20, alpha=0.6, label=label)

axes[0, 0].set_xlabel('Text Length (characters)', fontsize=12)
axes[0, 0].set_ylabel('Frequency', fontsize=12)
axes[0, 0].set_title('Text Length Distribution', fontsize=14, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Word count
for class_idx in misclass_df['actual_class'].unique():
    data = misclass_df[misclass_df['actual_class'] == class_idx]['word_count']
    label = 'False Positive' if class_idx == 0 else 'False Negative'
    axes[0, 1].hist(data, bins=20, alpha=0.6, label=label)

axes[0, 1].set_xlabel('Word Count', fontsize=12)
axes[0, 1].set_ylabel('Frequency', fontsize=12)
axes[0, 1].set_title('Word Count Distribution', fontsize=14, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# Confidence scores
axes[1, 0].scatter(misclass_df['prob_random'], misclass_df['prob_rc'], 
                   c=misclass_df['actual_class'], cmap='coolwarm', alpha=0.6, s=100)
axes[1, 0].plot([0, 1], [1, 0], 'k--', alpha=0.3, label='Decision Boundary')
axes[1, 0].set_xlabel('P(Random)', fontsize=12)
axes[1, 0].set_ylabel('P(RC Adaptation)', fontsize=12)
axes[1, 0].set_title('Prediction Probability Space', fontsize=14, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# Confidence distribution
axes[1, 1].hist(misclass_df['confidence'], bins=20, color='red', alpha=0.7)
axes[1, 1].axvline(misclass_df['confidence'].mean(), color='black', 
                   linestyle='--', linewidth=2, label=f"Mean: {misclass_df['confidence'].mean():.3f}")
axes[1, 1].set_xlabel('Confidence Score', fontsize=12)
axes[1, 1].set_ylabel('Frequency', fontsize=12)
axes[1, 1].set_title('Confidence Distribution of Errors', fontsize=14, fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('error_characteristics.png', dpi=300, bbox_inches='tight')
plt.show()

print("Visualization saved as 'error_characteristics.png'")

## 3. Embedding Similarity Analysis

In [ ]:
# Load USE model
print("Loading Universal Sentence Encoder...")
embed = hub.load("./USEmodel")
print("✓ Model loaded")

# Generate embeddings for misclassified texts
print("\nGenerating embeddings...")
misclass_texts = misclass_df['text'].tolist()
embeddings = embed(misclass_texts)

print(f"✓ Generated embeddings: {embeddings.shape}")

In [ ]:
# Compute pairwise similarities
similarity_matrix = cosine_similarity(embeddings)

print("\nSimilarity Matrix:")
print(f"Shape: {similarity_matrix.shape}")
print(f"\nSummary statistics:")
print(f"  Mean similarity: {similarity_matrix.mean():.4f}")
print(f"  Median similarity: {np.median(similarity_matrix):.4f}")
print(f"  Min similarity: {similarity_matrix[similarity_matrix != 1.0].min():.4f}")
print(f"  Max similarity: {similarity_matrix[similarity_matrix != 1.0].max():.4f}")

# Visualize similarity matrix
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(similarity_matrix, cmap='YlOrRd', aspect='auto')
ax.set_xlabel('Error Example Index', fontsize=12)
ax.set_ylabel('Error Example Index', fontsize=12)
ax.set_title('Embedding Similarity Matrix for Misclassified Examples', 
             fontsize=14, fontweight='bold')
plt.colorbar(im, ax=ax, label='Cosine Similarity')
plt.tight_layout()
plt.savefig('error_similarity_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("Visualization saved as 'error_similarity_matrix.png'")

## 4. Detailed Example Inspection

In [ ]:
# Inspect each misclassified example in detail
print("\nDETAILED ERROR INSPECTION")
print("=" * 80)

for idx, row in misclass_df.iterrows():
    actual = 'Random' if row['actual_class'] == 0 else 'RC Adaptation'
    predicted = 'Random' if row['predicted_class'] == 0 else 'RC Adaptation'
    error_type = 'False Positive' if row['actual_class'] == 0 else 'False Negative'
    
    print(f"\nExample {idx + 1}:")
    print("-" * 80)
    print(f"Error Type: {error_type}")
    print(f"  Actual: {actual}")
    print(f"  Predicted: {predicted}")
    print(f"  Confidence: {row['confidence']:.4f}")
    print(f"  P(Random): {row['prob_random']:.4f}")
    print(f"  P(RC): {row['prob_rc']:.4f}")
    print(f"\nText Statistics:")
    print(f"  Length: {row['text_length']:,} characters")
    print(f"  Words: {row['word_count']:,}")
    print(f"  Avg word length: {row['avg_word_length']:.2f}")
    print(f"\nText Preview (first 500 chars):")
    print(row['text'][:500])
    print("...")
    print()

## 5. Patterns and Insights

In [ ]:
# Generate insights report
insights_report = f"""
{'='*80}
ERROR ANALYSIS INSIGHTS
{'='*80}

SUMMARY:
{'-'*80}
Total misclassified: {len(misclass_df)}
  - False Positives: {len(misclass_df[misclass_df['actual_class'] == 0])}
  - False Negatives: {len(misclass_df[misclass_df['actual_class'] == 1])}

CONFIDENCE ANALYSIS:
{'-'*80}
Mean confidence: {misclass_df['confidence'].mean():.4f}
Std deviation: {misclass_df['confidence'].std():.4f}
Min confidence: {misclass_df['confidence'].min():.4f}
Max confidence: {misclass_df['confidence'].max():.4f}

TEXT CHARACTERISTICS:
{'-'*80}
Average text length: {misclass_df['text_length'].mean():,.0f} characters
Average word count: {misclass_df['word_count'].mean():,.0f} words
Average word length: {misclass_df['avg_word_length'].mean():.2f} characters

EMBEDDING SIMILARITY:
{'-'*80}
Mean inter-error similarity: {similarity_matrix[similarity_matrix != 1.0].mean():.4f}

KEY OBSERVATIONS:
{'-'*80}
1. Model errors are relatively rare ({len(misclass_df)} examples)
2. Error confidence is {'high' if misclass_df['confidence'].mean() > 0.75 else 'moderate'}
   (mean: {misclass_df['confidence'].mean():.2%})
3. {len(misclass_df[misclass_df['actual_class'] == 1])} RC adaptations were missed
4. {len(misclass_df[misclass_df['actual_class'] == 0])} random texts were falsely flagged

RECOMMENDATIONS:
{'-'*80}
- Review misclassified texts manually for data quality issues
- Consider edge cases in model training
- Investigate if errors share common linguistic patterns
- Evaluate if additional features could help boundary cases

{'='*80}
"""

print(insights_report)

# Save report
with open('error_analysis_insights.txt', 'w') as f:
    f.write(insights_report)

print("\n✓ Insights report saved to 'error_analysis_insights.txt'")

## Conclusion

This error analysis provides insights into the model's failure cases. The extremely low error rate (3 false negatives in the original results) suggests the model generalizes very well, but understanding these edge cases helps improve future iterations.